## Data Loading and Cleaning for Experiment

In [1]:
import math
import numpy as np
import pandas as pd
import os
from dotenv import load_dotenv
from huggingface_hub import login
import gc
import torch
from tqdm import tqdm
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
import ast

In [2]:
# get credentials from .env file and login to hugging face so we can run models
env_path = os.path.join( os.path.expanduser('~'), 'Documents', '.env')
load_dotenv(env_path)
hf_token = os.environ.get('HUGGINGFACE_API_TOKEN')
login(hf_token)

In [3]:
# ensure cache is empty so we can run
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
torch.cuda.empty_cache()

In [4]:
# grab data from path
df_full = pd.read_csv('./data/all_prompts.csv', index_col = 0)
df_full.head()

,text,source,label
0,"Federal law supersedes state law, and cannabis...",Bloom-7B,1
1,Miles feels restless after working all day. He...,Bloom-7B,1
2,So first of I am danish. That means that I fol...,Bloom-7B,1
3,In this paper we present a novel rule-based ap...,Bloom-7B,1
4,"Most social progressives, love democracy, and ...",Bloom-7B,1


In [5]:
# ensure data loaded correctly 
df_full.columns

Index(['text', 'source', 'label'], dtype='object')

In [6]:
# get distribution of labels
# note: human is 0, bot is 1
df_full['label'].value_counts()

label
1    508061
0    414523
Name: count, dtype: int64

In [7]:
# subsample since the number of rows is extremely large
SAMPLE_SIZE = 10000
RANDOM_SEED = 213

# note we equally; we are assuming that it is equally likely that text is bot generated or human (may not actually be the case)
df = (
    df_full
    .groupby('label', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), SAMPLE_SIZE // 2), random_state=RANDOM_SEED))
    .reset_index(drop=True)
)

C:\Users\Chris\AppData\Local\Temp\ipykernel_14324\1576568630.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), SAMPLE_SIZE // 2), random_state=RANDOM_SEED))


In [8]:
print(f"Shape: {df.shape}")
print('\n')
print(f"Distribution: {df['label'].value_counts()}")

Shape: (10000, 3)


Distribution: label
0    5000
1    5000
Name: count, dtype: int64


In [9]:
# add structural columns to mirror original experiment and allow it to reference choices
df['choices'] = [['Human-written', 'AI-generated']] * df.shape[0]
df['subject'] = 'AI_text_detection'
df['answerKey'] = df['label'].map({0: 'A', 1:'B'}) # remember: human = 0, bot = 1; this means human = A, bot = B

In [10]:
# define helper function for later; enable changing the max_length layer 
PROMPT_OVERHEAD = 150  # tokens for system prompt + chat template + instruction

def get_dynamic_max_length(text):
    char_len = len(text)
    if char_len < 500: # short texts: tweets, comments, sentences
        return 256 + PROMPT_OVERHEAD
    elif char_len < 1500: # medium texts: paragraphs, social media posts
        return 512 + PROMPT_OVERHEAD
    elif char_len < 3000: # long texts: articles, blog posts
        return 768 + PROMPT_OVERHEAD
    else: # very long texts: essays, academic papers
        return 1024 + PROMPT_OVERHEAD

In [11]:
# see how long it will take per
df['max_length_used'] = df['text'].apply(get_dynamic_max_length)
df['max_length_used'].value_counts()

max_length_used
1174    3101
406     2683
662     2286
918     1930
Name: count, dtype: int64

In [12]:
df.head()

,text,source,label,choices,subject,answerKey,max_length_used
0,"After the American Revolutionary War, the numb...",Human,0,"[Human-written, AI-generated]",AI_text_detection,A,662
1,Apple Company: Problems and Solutions Essay Ex...,Human,0,"[Human-written, AI-generated]",AI_text_detection,A,1174
2,I don't like the physical location of the Fies...,Human,0,"[Human-written, AI-generated]",AI_text_detection,A,918
3,Post-Traumatic Stress Disorder in Veterans Res...,Human,0,"[Human-written, AI-generated]",AI_text_detection,A,1174
4,Assessment Informed Instruction Essay\n\nAs I ...,Human,0,"[Human-written, AI-generated]",AI_text_detection,A,1174


## Initial Generator Policy Functions

In [13]:
# add system prompt to ensure directions are clear and consistent
SYSTEM_PROMPT = (
    "You are a text classification assistant. "
    "When given a text, you must classify it as either human-written or AI-generated. "
    "You always respond with only a single letter: A or B. "
    "You never explain your answer or add any other text."
)

In [44]:
# wraps prompt in a model's expected format to ensure models get the correct prompt in its intended format
# falls back to raw prompt if no chat template exists
def apply_chat_template(tokenizer, prompt_text, system_prompt = None):
    if tokenizer.chat_template is not None:
        messages = []
        
        # Try with system role first
        if system_prompt:
            try:
                test_messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt_text}
                ]
                return tokenizer.apply_chat_template(
                    test_messages,
                    tokenize=False,
                    add_generation_prompt=True
                )
            except Exception:
                # Model doesn't support system role — prepend into user message
                combined = f"{system_prompt}\n\n{prompt_text}"
                messages = [{"role": "user", "content": combined}]
        else:
            messages = [{"role": "user", "content": prompt_text}]
            
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    return prompt_text

In [45]:
# truncates raw text to maximum number of tokens before building the prompt; this will ensure the instruction is never cut off
def truncate_text(text, tokenizer, max_text_tokens = 800):
    tokens = tokenizer(text, return_tensors="pt").input_ids[0]
    if len(tokens) <= max_text_tokens:
        return text
    return tokenizer.decode(tokens[:max_text_tokens], skip_special_tokens=True)

In [46]:
# ensures underscores are removed so tokens are reliably separated by strings for the subject (in this case it is always "AI_text_detection")
def format_subject(subject):
    return " ".join(subject.split("_"))

In [47]:
# # builds generator prompt
# # get correct = True -> prompt asks for the corect label (human or AI)
# # get correct = False -> prompt asks for the incorrect label

# def build_generator_prompt(subject, target_question, target_choices, get_correct):

#     prompt = "The following are texts for {}. Determine whether each text is human-written or AI-generated.\n\n".format(
#         format_subject(subject)
#     )
#     prompt += f"{target_question}"
#     for i, c in enumerate(target_choices):
#         prompt += "\n{}. {}".format(chr(65 + i), c)

#     if get_correct:
#         prompt += "\nAnswer:"
#     else:
#         prompt += "\nIncorrect Answer:"
#     return prompt

In [48]:
# builds generator prompt differently from above; this produced better results
def build_generator_prompt(subject, target_text, target_choices, get_correct):
    if get_correct:
        return (
            f"Text:\n{target_text}\n\n"
            "Is the above text human-written or AI-generated?\n"
            "A. Human-written\n"
            "B. AI-generated\n"
            "Answer with only A or B."
        )
    else:
        return (
            f"Text:\n{target_text}\n\n"
            "What is the INCORRECT label for the above text?\n"
            "A. Human-written\n"
            "B. AI-generated\n"
            "Answer with only A or B."
        )

In [49]:
# extracts next-token logits for each choice letter and performs softmax
# returns dict: {A:p_A, B:p_B}
def get_generator_answer_probs(model, tokenizer, prompt_text, choices_list, max_length=512):
    formatted_prompt = apply_chat_template(tokenizer, prompt_text, system_prompt=SYSTEM_PROMPT)
    
    input_ids = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation="longest_first",
        max_length=max_length
    ).input_ids.to(model.device)

    with torch.no_grad():
        logits = model(input_ids=input_ids).logits[0, -1]

    choices = [chr(65 + i) for i in range(len(choices_list))]
    choice_logits = [
        logits[tokenizer(letter).input_ids[-1]].item()
        for letter in choices
    ]

    probs = torch.nn.functional.softmax(
        torch.tensor(choice_logits, device=model.device).float(), dim=0
    ).detach().cpu().numpy()

    return {choice: float(prob) for choice, prob in zip(choices, probs)}

In [50]:
# gets the probabilities for a given prompt
def generator_probs(subject, text, choices_list, get_correct, model, tokenizer, max_length=512):
    prompt = build_generator_prompt(subject, text, choices_list, get_correct)
    return get_generator_answer_probs(model, tokenizer, prompt, choices_list, max_length)

In [51]:
# returns dictionary with initial generator probabilities for the prompts
def get_initial_generator_probs(row, model, tokenizer):
    text       = truncate_text(row["text"], tokenizer, max_text_tokens=800)
    max_length = 1024   # hard ceiling, text is already pre-truncated
    gen_init   = {"correct": {}, "incorrect": {}}
    y_list     = row["choices"]
    subject    = row["subject"]

    for v in [True, False]:
        choice_probs = generator_probs(subject, text, y_list, v, model, tokenizer, max_length)
        key = "correct" if v else "incorrect"
        for letter, prob in choice_probs.items():
            gen_init[key][letter] = prob
    return gen_init

## Initial Discriminator Policies

In [52]:
# # builds the prompt as closely to the original experiment
# def build_discriminator_prompt(subject, text, proposed_label):
#     prompt = (
#         f"You are an expert evaluator of questions about {format_subject(subject)}.\n"
#         f"Determine if the proposed label is correct. Output ONLY 'A' or 'B'.\n"
#         f"Text: {text}\n"
#         f"Proposed Label: {proposed_label}\n\n"
#         f"Is this label correct? Respond ONLY with:\n"
#         f"A. Correct\n"
#         f"B. Incorrect\n\n"
#         f"Answer:"
#     )
#     return prompt

In [53]:
# the above prompt yielded extremely imbalanced results 
# rerunninng with a different prompt
# ensures instructions comes after text so the model reads it last
def build_discriminator_prompt(subject, text, proposed_label=None):
    """
    Instruction comes AFTER the text so the model reads it last,
    right before generating its response.
    """
    return (
        f"Text:\n{text}\n\n"
        "Is the above text human-written or AI-generated?\n"
        "A. Human-written\n"
        "B. AI-generated\n"
        "Answer with only A or B."
    )

In [54]:
# above function worked with old prompt
# function now changed to reflect structural changes to new prompt
def evaluate_answer_correctness(row, model, tokenizer):
    text      = truncate_text(row["text"], tokenizer, max_text_tokens=800)
    choices   = row["choices"]

    raw_prompt       = build_discriminator_prompt(subject=row["subject"], text=text)
    formatted_prompt = apply_chat_template(tokenizer, raw_prompt,
                                           system_prompt=SYSTEM_PROMPT)

    input_ids = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length = 1024   # hard ceiling as safety net
    ).input_ids.to(model.device)

    with torch.no_grad():
        logits = model(input_ids=input_ids).logits[0, -1]

    token_a = tokenizer("A").input_ids[-1]
    token_b = tokenizer("B").input_ids[-1]

    p_a, p_b = torch.nn.functional.softmax(
        torch.tensor([logits[token_a].item(), logits[token_b].item()]).float(),
        dim=0
    ).tolist()

    return {
        "A": {"correct": p_a, "incorrect": 1 - p_a},
        "B": {"correct": p_b, "incorrect": 1 - p_b}
    }

In [55]:
def get_initial_discriminator_probs(row, model, tokenizer):
    return evaluate_answer_correctness(row, model, tokenizer)

## Equilibrium Search

In [56]:
# method = 'generator': pick argmax_y pi_G(correct|y)
# method = 'discriminator': pick argmax_y pi_D(correct|y)
# note this function is rewritten but functionally the same as the one in the original 
def pick_answer(gen, disc, candidates, method = "generator"):
    if method == "generator":
        return max(candidates, key = lambda y: gen["correct"][y])
    else:
        return max(candidates, key = lambda y: disc[y]["correct"])

In [57]:
# numerically stable softmax over a 1D numpy array
def softmax(arr):
    m = np.max(arr)
    exp_vals = np.exp(arr - m)
    return exp_vals / np.sum(exp_vals)

In [58]:
# runs online mirror descent to find approximate nash equilibrium
def equilibrium_search(
    gen_init, disc_init,
    candidates,
    T = 20,
    eta_G = 0.1, eta_D = 0.1,
    lam_G = 0.1, lam_D = 0.1
):
    gen = {
        "correct": dict(gen_init["correct"]),
        "incorrect": dict(gen_init["incorrect"])
    }
    disc = {y: dict(disc_init[y]) for y in candidates}

    Qg = {
        "correct":   {y: 0.0 for y in candidates},
        "incorrect": {y: 0.0 for y in candidates}
    }
    Qd = {y: {"correct": 0.0, "incorrect": 0.0} for y in candidates}

    for t in range(1, T + 1):
        # 1) update Q accumulators
        for v in ["correct", "incorrect"]:
            for y in candidates:
                Qg[v][y] += (1.0 / (2.0 * t)) * disc[y][v]

        for y in candidates:
            for v in ["correct", "incorrect"]:
                Qd[y][v] += (1.0 / (2.0 * t)) * gen[v][y]

        # 2) update generator policy via mirror descent
        for v in ["correct", "incorrect"]:
            logits = [
                (Qg[v][y] + lam_G * math.log(gen_init[v][y] + 1e-12)) / (1 / eta_G + lam_G)
                for y in candidates
            ]
            new_probs = softmax(np.array(logits))
            for i, y in enumerate(candidates):
                gen[v][y] = new_probs[i]

        # 3) update discriminator policy via mirror descent
        for v in ["correct", "incorrect"]:
            logits = [
                (Qd[y][v] + lam_D * math.log(disc_init[y][v] + 1e-12)) / (1 / eta_D + lam_D)
                for y in candidates
            ]
            new_probs = softmax(np.array(logits))
            for i, y in enumerate(candidates):
                disc[y][v] = new_probs[i]

    return gen, disc

## Load Model

In [59]:
def load_model(model_name):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype = torch.float16,
        load_in_8bit = False,
        low_cpu_mem_usage = True,
        device_map = "cuda",
        trust_remote_code = True
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    return model, tokenizer

In [60]:
# mirrors subcategory_df_function from original paper
# for each row:
    # computes initial discriminator policy
    # computes initial generator policy
    # runs the equilibrium search
    # picks the final answers

def botdetect_policy_function(model, tokenizer, df):
    category_df = df.copy()

    gen_answer = []
    disc_answer = []
    gen_init_answer = []
    disc_init_answer = []
    disc_init_policy = []
    gen_init_policy = []
    disc_final_policy_consensus = []
    gen_final_policy_consensus = []

    for _, row in tqdm(category_df.iterrows(), total=len(category_df)):

        # discriminator init
        disc_init = get_initial_discriminator_probs(row, model, tokenizer)
        disc_init_policy.append(disc_init)
        gc.collect()
        torch.cuda.empty_cache()

        # generator init
        gen_init = get_initial_generator_probs(row, model, tokenizer)
        gen_init_policy.append(gen_init)
        gc.collect()
        torch.cuda.empty_cache()

        # initial answers (pre-equilibrium)
        gen_init_answer.append(max(gen_init["correct"], key=gen_init["correct"].get))
        disc_init_answer.append(max(disc_init, key=lambda choice: disc_init[choice]["correct"]))

        # candidates: binary ['A', 'B']
        candidates = [chr(65 + i) for i in range(len(row["choices"]))]

        # equilibrium search (identical to original)
        gen_final, disc_final = equilibrium_search(
            gen_init, disc_init, candidates,
            T = 20, eta_G = 0.1, eta_D = 0.1, lam_G = 0.1, lam_D = 0.1
        )
        disc_final_policy_consensus.append(disc_final)
        gen_final_policy_consensus.append(gen_final)

        # final answers (post-equilibirum)
        best_answer_g = pick_answer(gen_final, disc_final, candidates, method="generator")
        best_answer_d = pick_answer(gen_final, disc_final, candidates, method="discriminator")
        gen_answer.append(best_answer_g)
        disc_answer.append(best_answer_d)

    # output dataframe
    category_df["gen_init_answer"] = gen_init_answer
    category_df["disc_init_answer"] = disc_init_answer
    category_df["gen_answer"] = gen_answer
    category_df["disc_answer"] = disc_answer
    category_df["disc_init_policy"] = disc_init_policy
    category_df["gen_init_policy"] = gen_init_policy
    category_df["disc_final_policy_consensus"] = disc_final_policy_consensus
    category_df["gen_final_policy_consensus"] = gen_final_policy_consensus

    return category_df

In [61]:
# define function to perform VRAM cleanup between models to prevent errors between running models
def safe_model_cleanup():
    global model_d, tokenizer_d
    if 'model_d' in globals() and model_d is not None:
        model_d.cpu()  # move weights off GPU first
        del model_d
        model_d = None
    if 'tokenizer_d' in globals() and tokenizer_d is not None:
        del tokenizer_d
        tokenizer_d = None
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()  # wait for all CUDA ops to finish
    print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

## Sanity Check Before Running Initial Policies

In [62]:
safe_model_cleanup()
model_d, tokenizer_d = load_model("01-ai/Yi-1.5-9B-Chat")

df_test = df.sample(n=50, random_state=42).reset_index(drop=True)
temp_test = botdetect_policy_function(model_d, tokenizer_d, df_test)

print("True label distribution:")
print(df_test['label'].value_counts())
print("\ndisc_init_answer distribution:")
print(temp_test['disc_init_answer'].value_counts())
print("\nAccuracy:")
print(f"  {(temp_test['disc_init_answer'] == temp_test['answerKey']).mean():.2%}")

VRAM free: 24.01 GB


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:45<00:00,  1.09it/s]

True label distribution:
label
0    27
1    23
Name: count, dtype: int64

disc_init_answer distribution:
disc_init_answer
A    32
B    18
Name: count, dtype: int64

Accuracy:
  58.00%


In [63]:
# parse the disc_init_policy to see confidence levels
def parse_policy(p):
    if isinstance(p, str):
        return ast.literal_eval(p)
    return p

temp_test['disc_init_policy_parsed'] = temp_test['disc_init_policy'].apply(parse_policy)
temp_test['p_A'] = temp_test['disc_init_policy_parsed'].apply(lambda x: x['A']['correct'])
temp_test['p_B'] = temp_test['disc_init_policy_parsed'].apply(lambda x: x['B']['correct'])

print("Average P(A=human) by true label:")
print(temp_test.groupby('label')[['p_A', 'p_B']].mean())

Average P(A=human) by true label:
            p_A       p_B
label                    
0      0.620638  0.379362
1      0.578429  0.421571


In [68]:
# ensure prompt went through at end
test_row = df_test.iloc[3]
truncated = truncate_text(test_row["text"], tokenizer_d, max_text_tokens=800)
raw_prompt = build_discriminator_prompt(subject = test_row["subject"], text = truncated)
formatted  = apply_chat_template(tokenizer_d, raw_prompt, system_prompt=SYSTEM_PROMPT)

total_tokens = tokenizer_d(formatted, return_tensors="pt").input_ids.shape[1]
print(f"Total tokens after text truncation: {total_tokens}")

print("\nLAST 200 CHARS OF FORMATTED PROMPT:\n")
print(formatted[-200:])

Total tokens after text truncation: 896

LAST 200 CHARS OF FORMATTED PROMPT:

an airspace of 800, 000 M 3 (Mornington Peninsula Shire 2

Is the above text human-written or AI-generated?
A. Human-written
B. AI-generated
Answer with only A or B.<end_of_turn>
<start_of_turn>model



## Run Initial Policies

In [36]:
# Model 1: Qwen2.5-7B-Instruct
safe_model_cleanup()
model_d, tokenizer_d = load_model("Qwen/Qwen2.5-7B-Instruct")
temp_df_llama = botdetect_policy_function(model_d, tokenizer_d, df)

file_path = 'InitialPolicyData/botdetect_policy_df_Qwen25_7b.csv'
temp_df_llama.to_csv(file_path, index=False)
print(f"Saved: {file_path}")

VRAM free: 24.01 GB


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████| 10000/10000 [1:17:22<00:00,  2.15it/s]


Saved: InitialPolicyData/botdetect_policy_df_Qwen25_7b.csv


In [37]:
# Model 2: Mistral-7B-Instruct-v0.2
safe_model_cleanup()
model_d, tokenizer_d = load_model("mistralai/Mistral-7B-Instruct-v0.2")
temp_df_mistral = botdetect_policy_function(model_d, tokenizer_d, df)

file_path = 'InitialPolicyData/botdetect_policy_df_Mistral_7B.csv'
temp_df_mistral.to_csv(file_path, index=False)
print(f"Saved: {file_path}")

VRAM free: 24.01 GB


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████| 10000/10000 [1:19:54<00:00,  2.09it/s]


Saved: InitialPolicyData/botdetect_policy_df_Mistral_7B.csv


In [65]:
# Model 3: Gemma-7b-it
safe_model_cleanup()
model_d, tokenizer_d = load_model("google/gemma-7b-it")
temp_df_gemma = botdetect_policy_function(model_d, tokenizer_d, df)

file_path = 'InitialPolicyData/botdetect_policy_df_gemma_7b.csv'
temp_df_gemma.to_csv(file_path, index=False)
print(f"Saved: {file_path}")

VRAM free: 24.01 GB


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████| 10000/10000 [2:25:50<00:00,  1.14it/s]


Saved: InitialPolicyData/botdetect_policy_df_gemma_7b.csv


In [38]:
# Model 4: Yi-1.5-9B-Chat
safe_model_cleanup()
model_d, tokenizer_d = load_model("01-ai/Yi-1.5-9B-Chat")
temp_df_yi = botdetect_policy_function(model_d, tokenizer_d, df)

file_path = 'InitialPolicyData/botdetect_policy_df_Yi_9B.csv'
temp_df_yi.to_csv(file_path, index=False)
print(f"Saved: {file_path}")

VRAM free: 24.01 GB


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████| 10000/10000 [1:55:58<00:00,  1.44it/s]


Saved: InitialPolicyData/botdetect_policy_df_Yi_9B.csv


In [39]:
# Model 5: Granite-3.3-8b-base
safe_model_cleanup()
model_d, tokenizer_d = load_model("ibm-granite/granite-3.3-8b-base")
temp_df_granite = botdetect_policy_function(model_d, tokenizer_d, df)

file_path = 'InitialPolicyData/botdetect_policy_df_granite_8b.csv'
temp_df_granite.to_csv(file_path, index=False)
print(f"Saved: {file_path}")

VRAM free: 24.01 GB


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████| 10000/10000 [1:55:12<00:00,  1.45it/s]


Saved: InitialPolicyData/botdetect_policy_df_granite_8b.csv


In [40]:
# Model 6: Zephyr-7b-beta
safe_model_cleanup()
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

model_d, tokenizer_d = load_model("HuggingFaceH4/zephyr-7b-beta")
temp_df_zephyr = botdetect_policy_function(model_d, tokenizer_d, df)

file_path = 'InitialPolicyData/botdetect_policy_df_zephyr_7B.csv'
temp_df_zephyr.to_csv(file_path, index=False)
print(f"Saved: {file_path}")

VRAM free: 24.01 GB


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████| 10000/10000 [1:57:21<00:00,  1.42it/s]


Saved: InitialPolicyData/botdetect_policy_df_zephyr_7B.csv


In [41]:
# Model 7: Deepseek-llm-7b-chat
safe_model_cleanup()
model_d, tokenizer_d = load_model("deepseek-ai/deepseek-llm-7b-chat")
temp_df_deepseek_llm = botdetect_policy_function(model_d, tokenizer_d, df)

file_path = 'InitialPolicyData/botdetect_policy_df_deepseek_llm_7b.csv'
temp_df_deepseek_llm.to_csv(file_path, index=False)
print(f"Saved: {file_path}")

VRAM free: 24.01 GB


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████| 10000/10000 [2:02:33<00:00,  1.36it/s]


Saved: InitialPolicyData/botdetect_policy_df_deepseek_llm_7b.csv


In [42]:
# Model 8: DeepSeek-R1-Distill-Qwen-7B
safe_model_cleanup()
model_d, tokenizer_d = load_model("deepseek-ai/DeepSeek-R1-Distill-Qwen-7B")
temp_df_deepseek_qwen = botdetect_policy_function(model_d, tokenizer_d, df)

file_path = 'InitialPolicyData/botdetect_policy_df_deepseek_r1_qwen_7b.csv'
temp_df_deepseek_qwen.to_csv(file_path, index=False)
print(f"Saved: {file_path}")

VRAM free: 24.01 GB


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x000001EFC8B665F0>>
Traceback (most recent call last):
  File "C:\Users\Chris\anaconda3\envs\ml\lib\site-packages\ipykernel\ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 
100%|██████████████████████████████████████████████████████████████████████████| 10000/10000 [2:07:13<00:00,  1.31it/s]


Saved: InitialPolicyData/botdetect_policy_df_deepseek_r1_qwen_7b.csv
